# Load test — Zarr NWB with AIND metadata

Only dependencies: `hdmf_zarr` (brings `NWBZarrIO`) + stdlib `json` + `numpy`. No custom class import, no extension registration.

In [3]:
from hdmf_zarr import NWBZarrIO
import json
import numpy as np

write_save_file = '/root/capsule/data/test_nwb/test.nwb.zarr'
write_save_file = '/root/capsule/data/test_nwb/behavior_749472_2025-01-09_13-56-02_combined.nwb.zarr'

In [4]:
# One IO, kept open for the rest of the notebook so lazy datasets stay accessible.
io = NWBZarrIO(write_save_file, 'r')
nwb = io.read()

# AIND metadata lives at /general/aind_metadata/json_data (LabMetaData containers
# sit directly under /general/, not under /general/lab_meta_data/).
raw = io.file['general/aind_metadata/json_data'][()]
if isinstance(raw, np.ndarray):
    raw = raw.item()
if isinstance(raw, bytes):
    raw = raw.decode()
meta = json.loads(raw)

print('metadata files:', list(meta.keys()))

metadata files: ['acquisition', 'data_description', 'instrument', 'metadata.nd', 'procedures', 'subject']


In [5]:
print('acquisition subject_id:', meta['acquisition'].get('subject_id'))
print('subject genotype:', meta['subject'].get('subject_details', {}).get('genotype'))

acquisition subject_id: 749472
subject genotype: None


## Trials

In [6]:
trials_df = nwb.trials.to_dataframe()
print(f'{len(trials_df)} rows, {len(trials_df.columns)} columns')
print('ragged sample right_reward_times[0:5]:', trials_df['right_reward_times'].head(5).tolist())
trials_df.head()

460 rows, 34 columns
ragged sample right_reward_times[0:5]: [array([], dtype=float64), array([], dtype=float64), array([], dtype=float64), array([0.624]), array([0.375])]


,start_time,stop_time,trial_type,animal_response,rewarded_historyL,rewarded_historyR,goCue_start_time,reward_outcome_time,bait_left,bait_right,...,ITI_duration,delay_max,delay_min,lick_lat,trial_ind,right_reward_times,left_reward_times,choice_time_trial,auto_manual_trial,extra_reward
id,,,,,,,,,,,,,,,,,,,,,
0,97.919,102.519,CSplus,0,False,False,99.419,99.713,0,0,...,0.5,1,1,0.294,0,[],[],0.294,False,False
1,102.519,112.820,CSplus,0,False,False,109.720,110.267,0,0,...,0.5,1,1,0.547,1,[],[],0.547,False,False
2,112.820,121.566,CSplus,0,False,False,118.466,118.678,0,0,...,0.5,1,1,0.212,2,[],[],0.212,False,False
3,121.566,125.883,CSplus,1,False,True,122.783,123.206,0,0,...,0.5,1,1,0.423,3,[0.6239999999999952],[],0.423,False,False
4,125.883,130.447,CSplus,1,False,True,127.347,127.521,0,0,...,0.5,1,1,0.174,4,[0.375],[],0.174,False,False


## Units

In [7]:
units_df = nwb.units.to_dataframe()
print(f'{len(units_df)} rows, {len(units_df.columns)} columns')
print('spike counts per unit (first 5):', [len(units_df['spike_times'].iloc[i]) for i in range(5)])
units_df.head()

AttributeError: 'NoneType' object has no attribute 'to_dataframe'

## Acquisition TimeSeries

In [ ]:
for name, ts in nwb.acquisition.items():
    n = len(ts.timestamps)
    print(f'{name}: {n} timestamps, unit={ts.unit}, range=[{ts.timestamps[0]}, {ts.timestamps[-1]}]')

left_lick_time: 1942 timestamps, unit=second, range=[7165785.370752, 7170453.627008]
left_reward_delivery_time: 131 timestamps, unit=second, range=[7165795.771488, 7170046.026496]
right_lick_time: 2741 timestamps, unit=second, range=[7165778.541696, 7171771.0448]
right_reward_delivery_time: 159 timestamps, unit=second, range=[7165905.725504, 7170339.078496]


## Processing modules

In [ ]:
for pm_name, pm in nwb.processing.items():
    print(f'{pm_name}:')
    for obj_name, obj in pm.data_interfaces.items():
        print(f'  {obj_name}: {type(obj).__name__}')

In [ ]:
nwb.lab_meta_data

{'aind_metadata': aind_metadata abc.AindMetadata at 0x140372540701920
 Fields:
   json_data: {"acquisition": {"acquisition_end_time": "2021-05-01T20:46:52.115000-04:00", "acquisition_start_time": "2021-05-01T19:48:01-04:00", "acquisition_type": "Behavioral foraging", "calibrations": [], "coordinate_system": null, "data_streams": [{"active_devices": ["Neuralynx Ephys Assembly", "Pupil camera", "Arduino", "Lick spout assembly"], "code": null, "configurations": [{"device_name": "Neuralynx Ephys Assembly", "manipulator": {"coordinate_system": {"axes": [{"direction": "Posterior_to_anterior", "name": "AP", "object_type": "Axis"}, {"direction": "Left_to_right", "name": "ML", "object_type": "Axis"}, {"direction": "Superior_to_inferior", "name": "SI", "object_type": "Axis"}], "axis_unit": "millimeter", "name": "BREGMA_ARI", "object_type": "Coordinate system", "origin": "Bregma"}, "device_name": "Drive system", "local_axis_positions": {"object_type": "Translation", "translation": [0, 0, 0]}, "object_type": "Manipulator config"}, "modules": [], "object_type": "Ephys assembly config", "probes": []}], "connections": [], "modalities": [{"abbreviation": "behavior", "name": "Behavior"}, {"abbreviation": "ecephys", "name": "Extracellular electrophysiology"}], "notes": "ephys recording session", "object_type": "Data stream", "stream_end_time": "2021-05-01T20:46:52.115000-04:00", "stream_start_time": "2021-05-01T19:48:01-04:00"}], "describedBy": "https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/acquisition.py", "ethics_review_id": ["MO19M432"], "experimenters": ["zhixiao su"], "instrument_id": "hopkins_295F_nlyx", "maintenance": [], "notes": "ephys recording session for subject ZS062. tetrode depth = 4.429. session_notes = 'Good'", "object_type": "Acquisition", "protocol_id": null, "schema_version": "2.0.34", "specimen_id": null, "stimulus_epochs": [{"active_devices": ["Speaker"], "code": {"container": null, "core_dependency": null, "input_data": null, "language": null, "language_version": null, "name": "dynamic-foraging-task", "object_type": "Code", "parameters": {"BlockBeta": "15", "BlockMax": "35", "BlockMin": "20", "DelayBeta": "0.0", "DelayMax": "1.0", "DelayMin": "1.0", "ITIBeta": "3.0", "ITIMax": "15.0", "ITIMin": "2.0", "LeftValue_volume": "2.50", "RightValue_volume": "2.50", "Task": "Uncoupled Without Baiting", "amplitude_db": 60, "frequency_unit": "hertz", "go_cue_frequency": 7500, "no_go_cue_frequency": 15000, "reward_probability": "0.1, 0.5, 0.9", "sample_frequency": 96000, "stimulus_duration_ms": 500}, "run_script": null, "url": "https://github.com/JeremiahYCohenLab/sueBehavior.git", "version": null}, "configurations": [{"device_name": "Speaker", "object_type": "Speaker config", "volume": 60, "volume_unit": "decibels"}], "curriculum_status": null, "notes": null, "object_type": "Stimulus epoch", "performance_metrics": {"object_type": "Performance metrics", "output_parameters": {}, "reward_consumed_during_epoch": "675.0", "reward_consumed_unit": "microliter", "trials_finished": 424, "trials_rewarded": 270, "trials_total": 424}, "stimulus_end_time": "2021-05-01T20:46:52.115000-04:00", "stimulus_modalities": ["Auditory"], "stimulus_name": "Behavioral foraging task", "stimulus_start_time": "2021-05-01T19:48:01-04:00", "training_protocol_name": null}], "subject_details": {"anaesthesia": null, "animal_weight_post": null, "animal_weight_prior": "24.1", "mouse_platform_name": "mouse_tube_foraging_hopkins", "object_type": "Acquisition subject details", "reward_consumed_total": "0.675", "reward_consumed_unit": "milliliter", "weight_unit": "gram"}, "subject_id": "ZS062"}, "data_description": {"creation_time": "2021-05-01T19:48:01-04:00", "data_level": "derived", "data_summary": "Behavioral ephys recording session for subject ZS062", "describedBy": "https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/data_description.py", "funding_source": [{"f